# Código para ler os sinais de EEG e separar os eletrodos em vetores

In [ ]:
import numpy as np
import matplotlib.pyplot as mp
#--------------------------------------------------------------------------------------------------------------------------
#Controles
for i in range(1, 21):
    dadosC = np.genfromtxt('Dados/Control_' + str(i) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
    if dadosC.ndim == 1:
        dadosC = dadosC.reshape(-1, 1)
    dadosC = dadosC[~np.isnan(dadosC).any(axis = 1)]
    dadosC = dadosC[int(dadosC.shape[0]/2-23000):int(dadosC.shape[0]/2+23000), :]

    #print("Controle " + str(i))
    #print(dadosC)
    #print("Tamanho: ", np.shape(dadosC))
    #print('\n')

    for j in range(0,19):
        np.savetxt('Eletrodos-controle/controle_' + str(i) + 'eletrodo_' + str(j+1) + '.txt', dadosC[...,:,j], newline='\n')


#Pacientes com MCI
for i in range(1, 38):
    dadosMCI = np.genfromtxt('Dados/Patient_' + str(i) + '_MCI.txt', delimiter = ' ', dtype = float, invalid_raise = False)
    if dadosMCI.ndim == 1:
        dadosMCI = dadosMCI.reshape(-1, 1)
    dadosMCI = dadosMCI[~np.isnan(dadosMCI).any(axis=1)]
    dadosMCI = dadosMCI[int(dadosMCI.shape[0]/2-23000):int(dadosMCI.shape[0]/2+23000), :]

    #print("Paciente com MCI " + str(i))
    #rint(dadosMCI)
    #print("Tamanho: ", np.shape(dadosMCI))
    #print('\n')

    for j in range (0,19):
        np.savetxt('Eletrodos-DCL/DCL_' + str(i) + 'eletrodo_' + str(j+1) + '.txt', dadosMCI[...,:,j], newline='\n')


#Pacientes com AD
for i in range(1, 49):
    dadosAD = np.genfromtxt('Dados/Patient_' + str(i) + '_AD.txt', delimiter = ' ', dtype = float, invalid_raise = False)
    if dadosAD.ndim == 1:
        dadosAD = dadosAD.reshape(-1, 1)
    dadosAD = dadosAD[~np.isnan(dadosAD).any(axis=1)]
    dadosAD = dadosAD[int(dadosAD.shape[0]/2-23000):int(dadosAD.shape[0]/2+23000), :]

    #print("Paciente com AD " + str(i))
    #print(dadosAD)
    #print("Tamanho: ", np.shape(dadosAD))
    #print('\n')

    for j in range (0,19):
        np.savetxt('Eletrodos-DA/DA_' + str(i) + 'eletrodo_' + str(j+1) + '.txt', dadosAD[...,:,j], newline='\n')

# Código da decomposição e reconstrução dos sinais com Coiflets 1

In [ ]:
#Decomposição do sinal e construção das ondas de todos os eletrodos e pacientes
!pip install PyWavelets
import os
import numpy as np
import pywt

# Parâmetros da decomposição
wavelet_type = 'coif1'
level = 6

def reconstruir_banda(data, banda, wavelet_type='coif1', level=6, axis=-1):
    """
    Reconstrói o sinal preservando somente os coeficientes correspondentes à banda desejada,
    zerando os demais.
    
    Parâmetros:
      data         : sinal original.
      banda        : 'delta', 'theta', 'alfa' ou 'beta'.
      wavelet_type : tipo da wavelet (padrão 'coif1').
      level        : nível de decomposição (padrão 6).
      axis         : eixo da decomposição (padrão -1).
    
    Retorna:
      sinal_reconstruido : sinal reconstruído para a banda escolhida.
    """
    # Realizar a decomposição
    coeffs = pywt.wavedec(data, wavelet_type, mode='symmetric', level=level, axis=axis)
    
    if banda.lower() == 'delta':
        # Delta: zera todos os detalhes, preservando apenas cA6
        for i in range(1, len(coeffs)):
            coeffs[i] = np.zeros_like(coeffs[i])
    
    elif banda.lower() == 'theta':
        # Theta: zera a aproximação e todos os detalhes, exceto cD6 (coeffs[1] ou coeffs[-6])
        coeffs[0] = np.zeros_like(coeffs[0])
        coeffs[-1] = np.zeros_like(coeffs[-1])  # cD1
        coeffs[-2] = np.zeros_like(coeffs[-2])  # cD2
        coeffs[-3] = np.zeros_like(coeffs[-3])  # cD3
        coeffs[-4] = np.zeros_like(coeffs[-4])  # cD4
        coeffs[-5] = np.zeros_like(coeffs[-5])  # cD5
    
    elif banda.lower() == 'beta':
        # Beta: zera aproximação, cD6, cD5, cD3, cD2 e cD1; preserva cD4 (coeffs[-4])
        coeffs[0] = np.zeros_like(coeffs[0])
        coeffs[-1] = np.zeros_like(coeffs[-1])  # cD1
        coeffs[-2] = np.zeros_like(coeffs[-2])  # cD2
        coeffs[-3] = np.zeros_like(coeffs[-3])  # cD3
        # coeffs[-4] (cD4) permanece inalterado
        coeffs[-5] = np.zeros_like(coeffs[-5])  # cD5
        coeffs[-6] = np.zeros_like(coeffs[-6])  # cD6
    
    elif banda.lower() == 'alfa':
        # Alfa: zera aproximação, cD6, cD4, cD3, cD2 e cD1; preserva cD5 (coeffs[-5])
        coeffs[0] = np.zeros_like(coeffs[0])
        coeffs[-1] = np.zeros_like(coeffs[-1])  # cD1
        coeffs[-2] = np.zeros_like(coeffs[-2])  # cD2
        coeffs[-3] = np.zeros_like(coeffs[-3])  # cD3
        coeffs[-4] = np.zeros_like(coeffs[-4])  # cD4
        # coeffs[-5] (cD5) permanece inalterado
        coeffs[-6] = np.zeros_like(coeffs[-6])  # cD6
    
    else:
        raise ValueError("Banda desconhecida! Use 'delta', 'theta', 'alfa' ou 'beta'.")
    
    # Reconstruir o sinal a partir dos coeficientes modificados
    sinal_reconstruido = pywt.waverec(coeffs, wavelet_type, mode='symmetric', axis=axis)
    return sinal_reconstruido

# Criar as pastas de saída, se não existirem.
categorias = ['controle', 'DCL', 'DA']
bandas = ['Alfa', 'Beta', 'Theta', 'Delta']

num_eletrodos = 19

# --------------------------
# Grupo CONTROLE (pacientes 1 a 20)
# --------------------------
for e in range(1, num_eletrodos + 1):
    for p in range(1, 21):
        file_path = f'Eletrodos-controle/controle_{p}eletrodo_{e}.txt'
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}")
            continue
        # Carregar o sinal
        data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
        
        # Para cada banda, reconstrói o sinal e salva na pasta correspondente
        for banda in bandas:
            sinal_banda = reconstruir_banda(data, banda, wavelet_type, level, axis=-1)
            out_path = os.path.join(banda, "Ondas controle", f"controle_{p}eletrodo_{e}.txt")
            np.savetxt(out_path, sinal_banda)

# --------------------------
# Grupo DCL (pacientes 21 a 57)
# --------------------------
for e in range(1, num_eletrodos + 1):
    for p in range(21, 58):
        file_path = f'Eletrodos-DCL/DCL_{p-20}eletrodo_{e}.txt'
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}")
            continue
        data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
        for banda in bandas:
            sinal_banda = reconstruir_banda(data, banda, wavelet_type, level, axis=-1)
            out_path = os.path.join(banda, "Ondas DCL", f"DCL_{p-20}eletrodo_{e}.txt")
            np.savetxt(out_path, sinal_banda)

# --------------------------
# Grupo DA (pacientes 58 a 105)
# --------------------------
for e in range(1, num_eletrodos + 1):
    for p in range(58, 106):
        file_path = f'Eletrodos-DA/DA_{p-57}eletrodo_{e}.txt'
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}")
            continue
        data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
        for banda in bandas:
            sinal_banda = reconstruir_banda(data, banda, wavelet_type, level, axis=-1)
            out_path = os.path.join(banda, "Ondas DA", f"DA_{p-57}eletrodo_{e}.txt")
            np.savetxt(out_path, sinal_banda)

# Código do mapeamento em GQ

In [ ]:
#Mapeamento por grupo de pacientes
import os
import numpy as np
import math
import matplotlib.pyplot as mp

def printmatrix_MA(q, MA, k, p, e):
    # Abrir o arquivo no modo de escrita
    with open(f"Matrizes de adjacência/DA/MA_DA_{p}_eletrodo_{e}_k_{k}.txt", "w") as output1:
        # Percorrer cada linha e coluna da matriz
        for i in range(q):
            for j in range(q):
                # Escrever o elemento no arquivo com um espaço
                output1.write(f"{MA[i][j]} ")
            # Adicionar uma quebra de linha no final de cada linha da matriz
            output1.write("\n")

def main():
    w = 150  # Diferenças temporais

    # Leitura da série temporal
    for p in range(1,49):
        for e in range(1,20):
            with open('Eletrodos-DA/DA_' + str(p) + 'eletrodo_' + str(e) +'.txt', "r") as file:
                v = np.array([float(line.strip()) for line in file])

            n = v.size  # Tamanho da série
            q = int(round(2 * n**(1/3)))  # Quantis
            v_quantiles = np.zeros(n, dtype=int)
            v_st_ord_quantiles = np.zeros((n, 2), dtype=float)
            MA = np.zeros((q, q), dtype=int)

            #Ordenação da série
            v_ord = np.sort(v)
            #print("Série Ordenada:")
            for i in range(n):
                akj=1
                #print(f"{v[i]} {v_ord[i]}")

            #Determinação dos quantis
            Q = np.zeros((q+1), dtype=int)
            for i in range (0, q+1):
                Q[i]=((n*i)/q)+v_ord[0]

            for i in range (q):
                aju=1
                #print(f"{Q[i]} | -{Q[i+1]}" + "\n")

            for i in range (q):
                aux=i+1
                for j in range(int(Q[i]-v_ord[0]), int(Q[i+1]-v_ord[0])):
                    v_st_ord_quantiles[j][0]=v_ord[j]
                    v_st_ord_quantiles[j][1]=aux

            #print("Séries ordenadas e quantis:\n")
            for i in range (0, n):
                #print(f"{v_st_ord_quantiles[i][0], v_st_ord_quantiles[i][1]}" + "\n")
                akj=1
            indices = np.searchsorted(v_st_ord_quantiles[:, 0], v)
            indices = np.clip(indices, 0, v_st_ord_quantiles.shape[0] - 1)  # Encontrar índices onde `v` corresponde em `v_st_ord_quantiles`
            mask = (indices < v_st_ord_quantiles.shape[0]) & (v_st_ord_quantiles[indices, 0] == v)  # Verificar correspondência exata

            #Atualizar valores nos arrays baseados no mapeamento
            v_quantiles[mask] = (v_st_ord_quantiles[indices[mask], 1] - 1).astype(int)
            v_st_ord_quantiles[indices[mask], 0] = 1000000.

            #print("Séries-quantis:\n")
            for j in range (0, n):
                akj=1
                #print(v_quantiles[j], "\n")

            vector = []
            #Construção da matriz de adjacência
            for k in range(1, w+1):
                MA = np.zeros((q, q), dtype=int)
                for i in range (n - k):
                    MA[v_quantiles[i]][v_quantiles[i+k]] = MA[v_quantiles[i]][v_quantiles[i+k]]+1
                #print(MA)
                printmatrix_MA(q, MA, k, p, e)

            print(f"Paciente {p}, Eletrodo {e} processado.")


if __name__ == "__main__":
    main()

# Códigos dos cálculos dos caracterizadores topológicos das matrizes de adjacência

## Coeficiente de agrupamento médio

In [ ]:
#Coeficiente de agrupamento
import numpy as np
import matplotlib.pyplot as mp
import networkx as nx
from time import time

w = 150

eletrodos = {}

for e in range(1,20):
    start_time_total = time()
    eletrodos[f"Eletrodo_{e}"] = np.zeros((105, w), dtype=float)
    for p in range(1,21):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/Controle/MA_C_' + str(p) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            W_normalizado = MA/(np.max(MA))
            W_1terço = np.float_power(W_normalizado, 1/3)
            WT_1terço = np.transpose(W_1terço)
            Ws_somados = W_1terço + WT_1terço
            W3 = np.linalg.matrix_power(Ws_somados, 3)
            W3ii = np.transpose(W3.diagonal())
            MA_T = np.transpose(MA)
            d_in = np.count_nonzero(MA_T, axis=1, keepdims = True)
            d_out = np.count_nonzero(MA, axis=1, keepdims = True)
            d_tot = d_in + d_out
            d_bilateral = np.count_nonzero((MA * MA_T), axis=1, keepdims = True)
            denominador = 2 * ((d_tot * (d_tot - 1)) - (2 * d_bilateral))
            C_vetor = np.divide((W3ii.reshape(72, 1)), denominador)
            C_médio = (np.sum(C_vetor))/(C_vetor.size)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = C_médio
    for p in range(21,58):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DCL/MA_DCL_' + str(p-20) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            RW_normalizado = MA/(np.max(MA))
            W_1terço = np.float_power(W_normalizado, 1/3)
            WT_1terço = np.transpose(W_1terço)
            Ws_somados = W_1terço + WT_1terço
            W3 = np.linalg.matrix_power(Ws_somados, 3)
            W3ii = np.transpose(W3.diagonal())
            MA_T = np.transpose(MA)
            d_in = np.count_nonzero(MA_T, axis=1, keepdims = True)
            d_out = np.count_nonzero(MA, axis=1, keepdims = True)
            d_tot = d_in + d_out
            d_bilateral = np.count_nonzero((MA * MA_T), axis=1, keepdims = True)
            denominador = 2 * ((d_tot * (d_tot - 1)) - (2 * d_bilateral))
            C_vetor = np.divide((W3ii.reshape(72, 1)), denominador)
            C_médio = (np.sum(C_vetor))/(C_vetor.size)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = C_médio
    for p in range(58,106):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DA/MA_DA_' + str(p-57) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            W_normalizado = MA/(np.max(MA))
            W_1terço = np.float_power(W_normalizado, 1/3)
            WT_1terço = np.transpose(W_1terço)
            Ws_somados = W_1terço + WT_1terço
            W3 = np.linalg.matrix_power(Ws_somados, 3)
            W3ii = np.transpose(W3.diagonal())
            MA_T = np.transpose(MA)
            d_in = np.count_nonzero(MA_T, axis=1, keepdims = True)
            d_out = np.count_nonzero(MA, axis=1, keepdims = True)
            d_tot = d_in + d_out
            d_bilateral = np.count_nonzero((MA * MA_T), axis=1, keepdims = True)
            denominador = 2 * ((d_tot * (d_tot - 1)) - (2 * d_bilateral))
            C_vetor = np.divide((W3ii.reshape(72, 1)), denominador)
            C_médio = (np.sum(C_vetor))/(C_vetor.size)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = C_médio
    np.savetxt('Caracterizadores/Coeficiente de agrupamento/Eletrodo ' + str(e) + '.txt', eletrodos[f"Eletrodo_{e}"], newline='\n')

    print(f"Eletrodo {e} concluído")
    end_time_total = time()
    t_total = end_time_total - start_time_total
    print(f"tempo total é {t_total} segundos")

## Reciprocidade

In [ ]:
#Reciprocidade
import numpy as np
import matplotlib.pyplot as mp
import networkx as nx
from time import time

start_time_total = time()

w = 150

eletrodos = {}

for e in range(1,20):
    eletrodos[f"Eletrodo_{e}"] = np.zeros((105, w), dtype=float)
    for p in range(1,21):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/Controle/MA_C_' + str(p) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Reciprocidade = nx.overall_reciprocity(Rede)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Reciprocidade
    for p in range(21,58):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DCL/MA_DCL_' + str(p-20) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Reciprocidade = nx.overall_reciprocity(Rede)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Reciprocidade
    for p in range(58,106):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DA/MA_DA_' + str(p-57) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Reciprocidade = nx.overall_reciprocity(Rede)
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Reciprocidade
    np.savetxt('Caracterizadores/Reciprocidade/Eletrodo ' + str(e) + '.txt', eletrodos[f"Eletrodo_{e}"], newline='\n')

end_time_total = time()
t_total = end_time_total - start_time_total
print(f"tempo total é {t_total} segundos")

## Grau de intermediação

In [ ]:
#Intermediação
import numpy as np
import matplotlib.pyplot as mp
import networkx as nx
from time import time

w = 150

eletrodos = {}

for e in range(1,6):
    start_time_total = time()
    eletrodos[f"Eletrodo_{e}"] = np.zeros((105, w), dtype=float)
    for p in range(1,21):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/Controle/MA_C_' + str(p) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            BC = nx.betweenness_centrality(Rede, k=int(N), weight='weight', endpoints=True)
            if __name__ == '__main__':
                total = sum(BC.values())
            Intermediação = total/4
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Intermediação
    for p in range(21,58):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DCL/MA_DCL_' + str(p-20) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            BC = nx.betweenness_centrality(Rede, k=int(N), weight='weight', endpoints=True)
            if __name__ == '__main__':
                total = sum(BC.values())
            Intermediação = total/4
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Intermediação
    for p in range(58,106):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DA/MA_DA_' + str(p-57) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            BC = nx.betweenness_centrality(Rede, k=int(N), weight='weight', endpoints=True)
            if __name__ == '__main__':
                total = sum(BC.values())
            Intermediação = total/4
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Intermediação
    np.savetxt('Caracterizadores/Intermediação/Eletrodo ' + str(e) + '.txt', eletrodos[f"Eletrodo_{e}"], newline='\n')

    print(f"Eletrodo {e} concluído")
    end_time_total = time()
    t_total = end_time_total - start_time_total
    print(f"tempo total é {t_total} segundos")

## Comprimento do menor caminho médio

In [ ]:
#Menor caminho médio
import numpy as np
import matplotlib.pyplot as mp
import networkx as nx
from time import time

w = 150

eletrodos = {}

for e in range(1,20):
    start_time_total = time()
    eletrodos[f"Eletrodo_{e}"] = np.zeros((105, w), dtype=float)
    for p in range(1,21):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/Controle/MA_C_' + str(p) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Menor_caminho_médio = nx.average_shortest_path_length(Rede, weight="weight", method='floyd-warshall')
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Menor_caminho_médio
    for p in range(21,58):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DCL/MA_DCL_' + str(p-20) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Menor_caminho_médio = nx.average_shortest_path_length(Rede, weight="weight", method='floyd-warshall')
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Menor_caminho_médio
    for p in range(58,106):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DA/MA_DA_' + str(p-57) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            Rede = nx.from_numpy_array(MA, parallel_edges=True, create_using=nx.DiGraph)
            Menor_caminho_médio = nx.average_shortest_path_length(Rede, weight="weight", method='floyd-warshall')
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Menor_caminho_médio
    np.savetxt('Caracterizadores/Menor caminho médio/Eletrodo ' + str(e) + '.txt', eletrodos[f"Eletrodo_{e}"], newline='\n')

    print(f"Eletrodo {e} concluído")
    end_time_total = time()
    t_total = end_time_total - start_time_total
    print(f"tempo total é {t_total} segundos")

## Comprimento do salto médio

In [ ]:
#Salto médio
import numpy as np
import matplotlib.pyplot as mp
import networkx as nx
from time import time

w = 150

eletrodos = {}

for e in range(1,20):
    start_time_total = time()
    eletrodos[f"Eletrodo_{e}"] = np.zeros((105, w), dtype=float)
    for p in range(1,21):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/Controle/MA_C_' + str(p) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            SUMA = MA.sum(axis=1)
            Pt = np.zeros(shape = MA.shape)
            G = np.zeros(shape = MA.shape)
            for i in range (0, int(N)):
                for j in range (0,int(N)):
                    Pt[j][i] = MA[i][j]/SUMA[i]
                    G[i][j] = abs(i - j)
            GPt = np.dot(G, Pt)
            tr = np.trace(GPt)
            Salto_médio = tr/N
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Salto_médio
    for p in range(21,58):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DCL/MA_DCL_' + str(p-20) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            SUMA = MA.sum(axis=1)
            Pt = np.zeros(shape = MA.shape)
            G = np.zeros(shape = MA.shape)
            for i in range (0, int(N)):
                for j in range (0,int(N)):
                    Pt[j][i] = MA[i][j]/SUMA[i]
                    G[i][j] = abs(i - j)
            GPt = np.dot(G, Pt)
            tr = np.trace(GPt)
            Salto_médio = tr/N
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Salto_médio
    for p in range(58,106):
        for k in range(1,w+1):
            MA = np.genfromtxt('Matrizes de adjacência/DA/MA_DA_' + str(p-57) + '_eletrodo_' + str(e) + '_k_' + str(k) + '.txt', delimiter = " ", dtype = float, invalid_raise = False)
            N = (MA.size)**(1/2)
            SUMA = MA.sum(axis=1)
            Pt = np.zeros(shape = MA.shape)
            G = np.zeros(shape = MA.shape)
            for i in range (0, int(N)):
                for j in range (0,int(N)):
                    Pt[j][i] = MA[i][j]/SUMA[i]
                    G[i][j] = abs(i - j)
            GPt = np.dot(G, Pt)
            tr = np.trace(GPt)
            Salto_médio = tr/N
            eletrodos[f"Eletrodo_{e}"][p-1][k-1] = Salto_médio
    np.savetxt('Caracterizadores/Salto médio/Eletrodo ' + str(e) + '.txt', eletrodos[f"Eletrodo_{e}"], newline='\n')

    print(f"Eletrodo {e} concluído")
    end_time_total = time()
    t_total = end_time_total - start_time_total
    print(f"tempo total é {t_total} segundos")

# Código do cálculo da dimensão fractal de Katz

In [ ]:
#Katz do sinal original
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import detrend

def katz_fractal_dimension(time_series):
    """
    Calcula a dimensão fractal de Katz para uma série temporal com base na metodologia fornecida.

    Parâmetros:
        time_series (array-like): A série temporal para cálculo da dimensão fractal.

    Retorno:
        fractal_dimension (float): A dimensão fractal calculada.
    """
    time_series = detrend(time_series)

    #Número total de pontos
    T = len(time_series)

    #Comprimento total do percurso (D)
    D = np.sum(np.sqrt(1 + np.diff(time_series)**2))

    #Distância máxima (d)
    d = max(
        np.sqrt((1 - i)**2 + (time_series[0] - time_series[i])**2)
        for i in range(1, T)
    )

    #Dimensão fractal de Katz
    fractal_dimension = np.log(T - 1) / np.log((d * T - d) / D)

    return fractal_dimension

if __name__ == "__main__":
    num_eletrodos = 19
    num_pacientes = 105

    fractal_dimensions = np.zeros((num_pacientes, num_eletrodos))

    for e in range(1, num_eletrodos + 1):
        for p in range(1, 21):
            fileC = np.genfromtxt(f'Eletrodos-controle/controle_{p}eletrodo_{e}.txt', delimiter=" ", dtype=float, invalid_raise=False)
            time_seriesC = np.array(fileC)

            fractal_dimension = katz_fractal_dimension(time_seriesC)

            fractal_dimensions[p - 1, e - 1] = fractal_dimension

        for p in range(21, 58):
            fileDCL = np.genfromtxt(f'Eletrodos-DCL/DCL_{p-20}eletrodo_{e}.txt', delimiter=" ", dtype=float, invalid_raise=False)
            time_seriesDCL = np.array(fileDCL)

            fractal_dimension = katz_fractal_dimension(time_seriesDCL)

            fractal_dimensions[p - 1, e - 1] = fractal_dimension

        for p in range(58, 106):
            fileDA = np.genfromtxt(f'Eletrodos-DA/DA_{p-57}eletrodo_{e}.txt', delimiter=" ", dtype=float, invalid_raise=False)
            time_seriesDA = np.array(fileDA)

            fractal_dimension = katz_fractal_dimension(time_seriesDA)

            fractal_dimensions[p - 1, e - 1] = fractal_dimension

        print(f"Eletrodo {e} concluído.")

    np.savetxt("dimensao_fractal_katz.csv", fractal_dimensions, delimiter=",", header=",".join([f"Eletrodo_{i}" for i in range(1, num_eletrodos + 1)]), comments="")

    print("Processamento concluído. Matriz salva em 'dimensao_fractal_katz.csv'.")

# Código do cálculo da entropia amostral quadrática

In [ ]:
#QSE do sinal original
import numpy as np
from scipy.spatial import cKDTree
import os
from joblib import Parallel, delayed

def zscore_normalization(signal):
    """
    Realiza a padronização Z-score de um sinal.
    """
    mean = np.mean(signal)
    std = np.std(signal)
    if std == 0:
        return signal  # evita divisão por zero se o desvio padrão for 0
    return (signal - mean) / std

def compute_qse_for_series(L, m, r_values):
    """
    Calcula a QSE para uma série temporal L para um valor de m (embedding dimension)
    e para cada valor de r em r_values.

    Para B utiliza templates de tamanho m e, para A, templates de tamanho m+1.
    Antes do cálculo, o sinal é padronizado via Z-score.

    Retorna uma lista com os valores de QSE correspondentes a cada r.
    """
    # Padronização da série temporal
    L = zscore_normalization(L)

    N = len(L)
    if N < m + 1:
        return [np.nan] * len(r_values)

    # Cria templates de dimensão m utilizando sliding_window_view
    X = np.lib.stride_tricks.sliding_window_view(L, window_shape=m)
    # Para B: usa os primeiros templates e todos os templates
    Q = X[:-1]
    R = X
    tree_Q = cKDTree(Q, leafsize=40)
    tree_R = cKDTree(R, leafsize=40)

    # Para A: templates de dimensão m+1
    Y = np.lib.stride_tricks.sliding_window_view(L, window_shape=m+1)
    tree_Y = cKDTree(Y, leafsize=40)

    qse_vals = []
    for r in r_values:
        if r <= 0:
            qse_vals.append(np.nan)
            continue
        # Conta os vizinhos para Q na árvore R e subtrai o auto-match
        B = tree_Q.count_neighbors(tree_R, r, p=np.inf) - len(Q)
        # Conta os vizinhos para a árvore de Y
        A = tree_Y.count_neighbors(tree_Y, r, p=np.inf) - len(Y)

        if A <= 0 or B <= 0:
            qse_vals.append(np.nan)
        else:
            qse_vals.append(-np.log(A / B) + np.log(2 * r))
    return qse_vals

def process_eletrode(e, r_values, m_values, num_r, num_pacientes):
    """
    Processa um eletrodo inteiro, lendo os arquivos de todos os grupos e
    montando a matriz (num_pacientes x (número de m_values * num_r)) para cada eletrodo.
    Retorna a matriz e o número do eletrodo.
    """
    # Cria a matriz para os 105 pacientes e para cada combinação
    matrix = np.full((num_pacientes, num_r * len(m_values)), np.nan)

    # Processa o grupo Controle
    for p in range(1, 21):
        file_path = f'Eletrodos-controle/controle_{p}eletrodo_{e}.txt'
        global_p = p  # número global do paciente
        if not os.path.exists(file_path):
            print(f"[Eletrodo {e}] Arquivo não encontrado: {file_path}")
            continue
        try:
            data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
            time_series = np.asarray(data).flatten()
        except Exception as ex:
            print(f"[Eletrodo {e}] Erro ao carregar {file_path}: {ex}")
            continue

        for i, m in enumerate(m_values):
            qse_list = compute_qse_for_series(time_series, m, r_values)
            start_col = i * num_r
            end_col = start_col + num_r
            matrix[global_p - 1, start_col:end_col] = qse_list

    # Processa o grupo DCL
    for p in range(21, 58):
        file_path = f'Eletrodos-DCL/DCL_{p-20}eletrodo_{e}.txt'
        global_p = p
        if not os.path.exists(file_path):
            print(f"[Eletrodo {e}] Arquivo não encontrado: {file_path}")
            continue
        try:
            data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
            time_series = np.asarray(data).flatten()
        except Exception as ex:
            print(f"[Eletrodo {e}] Erro ao carregar {file_path}: {ex}")
            continue

        for i, m in enumerate(m_values):
            qse_list = compute_qse_for_series(time_series, m, r_values)
            start_col = i * num_r
            end_col = start_col + num_r
            matrix[global_p - 1, start_col:end_col] = qse_list

    # Processa o grupo DA
    for p in range(58, 106):
        file_path = f'Eletrodos-DA/DA_{p-57}eletrodo_{e}.txt'
        global_p = p
        if not os.path.exists(file_path):
            print(f"[Eletrodo {e}] Arquivo não encontrado: {file_path}")
            continue
        try:
            data = np.genfromtxt(file_path, delimiter=" ", dtype=float, invalid_raise=False)
            time_series = np.asarray(data).flatten()
        except Exception as ex:
            print(f"[Eletrodo {e}] Erro ao carregar {file_path}: {ex}")
            continue

        for i, m in enumerate(m_values):
            qse_list = compute_qse_for_series(time_series, m, r_values)
            start_col = i * num_r
            end_col = start_col + num_r
            matrix[global_p - 1, start_col:end_col] = qse_list

    # Header para as colunas
    header_list = []
    for m in m_values:
        for r in r_values:
            header_list.append(f"m{m}_r{r:.2f}")
    header = ",".join(header_list)

    # Salvamento da matriz para o eletrodo
    output_folder = "Matrizes_qse"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    output_file = os.path.join(output_folder, f"qse_eletrodo_{e}.csv")
    np.savetxt(output_file, matrix, delimiter=",", header=header, comments="", fmt="%.6f")
    print(f"[Eletrodo {e}] Concluído e salvo em: {output_file}")
    return e, matrix

if __name__ == "__main__":
    # Configurações
    num_eletrodos = 19
    num_pacientes = 105
    r_values = np.arange(0.05, 1.05, 0.05)
    m_values = [1, 2]
    num_r = len(r_values)

    # Paraleliza o processamento de cada eletrodo
    # n_jobs=-1 para usar todos os núcleos disponíveis
    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(process_eletrode)(e, r_values, m_values, num_r, num_pacientes)
        for e in range(1, num_eletrodos + 1)
    )

    print("Processamento de todos os eletrodos concluído.")

# Código do SVM dos caracterizadores

In [ ]:
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Lista com os nomes dos eletrodos
nomes_eletrodos = ["Fp1", "F3", "C3", "P3", "O1", "F7", "T3", "T5", "Fz", "Cz", "Pz", "Fp2", "F4", "C4", "P4", "O2", "F8", "T4", "T6"]

# Sinais a serem analisados
sinais = ['Sinal original','Ondas alfa', 'Ondas beta', 'Ondas theta', 'Ondas delta']

# Labels dos pacientes (20 Controle, 37 DCL e 48 DA -> total 105)
labels = np.array([0] * 20 + [1] * 37 + [2] * 48)

# Loop para cada sinal
for sinal in sinais:
    print(f"\nIniciando análise SVM para o sinal: {sinal}...")
    
    # Novo diretório de entrada: "Caracterizadores/<Sinal>/caracterizador"
    input_dir = os.path.join("Caracterizadores", sinal, "Salto médio")
    
    # Diretório para salvar os resultados SVM para o sinal atual
    svm_output_dir = os.path.join("Resultados SVM", sinal)
    os.makedirs(svm_output_dir, exist_ok=True)
    
    # Dicionário para armazenar as acurácias de cada eletrodo e lista para os melhores resultados
    all_accuracies = {}
    max_accuracies = []
    
    # Para cada eletrodo (1 a 19)
    for e in range(1, 20):
        file_path = os.path.join(input_dir, f"Eletrodo_{e}.txt")
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}")
            continue
        
        # Carregar a matriz (dimensão: [105 x 150])
        matriz = np.loadtxt(file_path)
        
        # Parâmetros para o SVC
        param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': [0.001, 0.01, 0.1, 1],
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
        }
        
        k_folds = 5
        skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
        
        accuracies = []
        best_acc = 0
        best_k = 0
        
        # Testa cada coluna (característica) da matriz
        for k in range(matriz.shape[1]):
            X = matriz[:, k].reshape(-1, 1)
            fold_accuracies = []
            
            for train_idx, test_idx in skf.split(X, labels):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = labels[train_idx], labels[test_idx]
                
                grid = SVC(class_weight='balanced', random_state=42)
                grid.fit(X_train, y_train)

                y_pred = grid.predict(X_test)
                acc = accuracy_score(y_test, y_pred)
                fold_accuracies.append(acc)
            
            mean_acc = np.mean(fold_accuracies)
            accuracies.append(mean_acc)
            
            if mean_acc > best_acc:
                best_acc = mean_acc
                best_k = k + 1
        
        print(f"Eletrodo {e} ({nomes_eletrodos[e-1]}) concluído para o sinal: {sinal}")
        all_accuracies[nomes_eletrodos[e-1]] = accuracies
        max_accuracies.append([nomes_eletrodos[e-1], best_acc, best_k])
    
    # Salvar a matriz de melhores acurácias
    txt_output = os.path.join(svm_output_dir, "melhores_acuracias_Salto médio.txt")
    header = "Eletrodo,Acuracia_maxima,Indice_da_Feature"
    np.savetxt(txt_output, max_accuracies, fmt='%s', delimiter=",", header=header, comments='')
    print(f"Matriz de melhores acurácias salva como {txt_output}")

# Código do SVM das entropias quadráticas

In [ ]:
import os
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Lista com os nomes dos eletrodos
nomes_eletrodos = ["Fp1", "F3", "C3", "P3", "O1", "F7", "T3", "T5", "Fz", "Cz", "Pz", "Fp2", "F4", "C4", "P4", "O2", "F8", "T4", "T6"]

# Sinais a serem analisados
sinais = ['Sinal original', 'Ondas alfa', 'Ondas beta', 'Ondas theta', 'Ondas delta']

# Labels dos pacientes (20 Controle, 37 DCL e 48 DA -> total 105)
labels = np.array([0] * 20 + [1] * 37 + [2] * 48)

# Valores de m e r utilizados no cálculo do QSE
m_values = [1, 2]
r_values = np.arange(0.05, 1.05, 0.05)
num_r = len(r_values)

# Loop para cada sinal
for sinal in sinais:
    print(f"\nIniciando análise SVM para o sinal {sinal}...")

    # Definir a lógica para pasta e prefixo do arquivo
    if sinal.lower() == "sinal original":
        folder_base = "Sinal original"
        file_prefix = "sinal original"
    elif sinal.lower().startswith("ondas "):
        # Extrai a segunda palavra do sinal (ex.: "alfa", "beta", etc.)
        folder_base = sinal.split()[1].capitalize()  # ex: "Alfa", "Beta", etc.
        file_prefix = sinal.split()[1].lower()         # ex: "alfa", "beta", etc.
    else:
        folder_base = sinal.capitalize()
        file_prefix = sinal.lower()

    # Diretório de entrada no formato: Caracterizadores, dimensão fractal e qse/<Sinal>/qse_<folder_base>
    input_dir = os.path.join("Caracterizadores, dimensão fractal e qse", sinal, f"qse_{folder_base}")

    # Diretório para salvar os resultados SVM para o sinal atual
    svm_output_dir = os.path.join("Resultados SVM", sinal)
    os.makedirs(svm_output_dir, exist_ok=True)

    # Dicionário para armazenar as acurácias de cada eletrodo e lista para os melhores resultados
    all_accuracies = {}
    max_accuracies = []

    # Para cada eletrodo (1 a 19)
    for e in range(1, 20):
        file_path = os.path.join(input_dir, f"qse_{file_prefix}_eletrodo_{e}.csv")
        if not os.path.exists(file_path):
            print(f"Arquivo não encontrado: {file_path}")
            continue

        # Carregar a matriz
        try:
            matriz = np.loadtxt(file_path, delimiter=",", skiprows=1)
        except Exception as ex:
            print(f"Erro ao carregar {file_path}: {ex}")
            continue

        # Parâmetros para o SVC
        param_grid = {
            'C': [0.1, 1, 10, 100],
            'gamma': [0.001, 0.01, 0.1, 1],
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
        }

        k_folds = 5
        skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

        accuracies = []
        best_acc = 0
        best_feature_index = 0

        # Testa cada coluna (característica) da matriz
        for k in range(matriz.shape[1]):
            X = matriz[:, k].reshape(-1, 1)
            fold_accuracies = []

            for train_idx, test_idx in skf.split(X, labels):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = labels[train_idx], labels[test_idx]

                grid = SVC(class_weight='balanced', random_state=42)
                grid.fit(X_train, y_train)
                y_pred = grid.predict(X_test)
                acc = accuracy_score(y_test, y_pred)
                fold_accuracies.append(acc)

            mean_acc = np.mean(fold_accuracies)
            accuracies.append(mean_acc)

            if mean_acc > best_acc:
                best_acc = mean_acc
                best_feature_index = k

        # Determina os valores de m e r correspondentes ao índice da melhor feature
        if best_feature_index < num_r:
            best_m = m_values[0]
            best_r = r_values[best_feature_index]
        else:
            best_m = m_values[1]
            best_r = r_values[best_feature_index - num_r]

        print(f"Eletrodo {e} ({nomes_eletrodos[e-1]}) concluído para o sinal {sinal}")
        all_accuracies[nomes_eletrodos[e-1]] = accuracies
        max_accuracies.append([nomes_eletrodos[e-1], best_acc, best_m, best_r])

    # Salvar a matriz de melhores acurácias com os valores de m e r
    txt_output = os.path.join(svm_output_dir, "melhores_acuracias_qse.txt")
    header = "Eletrodo,Acuracia_maxima,m,r"
    np.savetxt(txt_output, max_accuracies, fmt='%s', delimiter=",", header=header, comments='')
    print(f"Matriz de melhores acurácias salva como {txt_output}")

# Código do SVM das dimensões fractais

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from matplotlib.cm import get_cmap

# Lista de sinais e nomes dos eletrodos
sinais = ['Sinal original', 'Ondas alfa', 'Ondas beta', 'Ondas theta', 'Ondas delta']
eletrodos_nomes = ["Fp1", "F3", "C3", "P3", "O1", "F7", "T3", "T5", "Fz", "Cz", "Pz", 
                   "Fp2", "F4", "C4", "P4", "O2", "F8", "T4", "T6"]

# Labels das classes: CONTROLE (20), DCL (37) e DA (48)
labels = np.array([0] * 20 + [1] * 37 + [2] * 48)

# Hiperparâmetros para otimização
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

# Configuração do k-fold
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# Diretório base para salvar os resultados
output_base_dir = "Resultados SVM"

# Diretório base para os arquivos de entrada
input_base_dir = "Caracterizadores, dimensão fractal e qse"

# Itera sobre cada sinal
for sinal in sinais:
    # Define a parte do nome do arquivo com base no sinal:
    # Para sinais que começam com "Ondas ", utiliza apenas o nome da onda (com inicial maiúscula)
    if sinal.startswith("Ondas "):
        file_name_part = sinal.split(" ")[-1].capitalize()
    else:
        file_name_part = sinal

    # Caminho do arquivo gerado no processamento da dimensão fractal
    input_file = os.path.join(input_base_dir, sinal, f"Dimensão fractal_{file_name_part}.csv")
    if not os.path.exists(input_file):
        print(f"Arquivo não encontrado: {input_file}")
        continue

    # Carrega a planilha
    df = pd.read_csv(input_file)

    # Dicionário para armazenar as acurácias de cada eletrodo
    accuracies = {}

    # Processa cada eletrodo
    for i, nome in enumerate(eletrodos_nomes):
        # Seleciona os dados do eletrodo e reformata para (n_amostras, 1)
        X = df.iloc[:, i].values.reshape(-1, 1)

        fold_accuracies = []

        for train_idx, test_idx in skf.split(X, labels):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = labels[train_idx], labels[test_idx]

            grid = SVC(class_weight='balanced', random_state=42)
            grid.fit(X_train, y_train)

            y_pred = grid.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            fold_accuracies.append(acc)

        mean_acc = np.mean(fold_accuracies)
        accuracies[nome] = mean_acc

    # Cria o diretório de saída para o sinal, se não existir
    output_dir = os.path.join(output_base_dir, sinal)
    os.makedirs(output_dir, exist_ok=True)
    
    # Salva os valores das acurácias em um arquivo TXT
    output_txt = os.path.join(output_dir, f"acuracias_{sinal.lower()}.txt")
    with open(output_txt, "w") as f:
        for nome, acc in accuracies.items():
            f.write(f"{nome}: {acc:.4f}\n")
    
    print(f"Processamento concluído para o sinal {sinal}. Resultados salvos em '{output_dir}'.")

# Código para fazer a figura da disposição dos eletrodos no escalpo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_scalp(ax, df, cond, electrode_positions, title_str):
    """
    Plota uma "cabeça" no ax, com os eletrodos posicionados de acordo com electrode_positions,
    coloridos conforme os valores de acurácia em df[cond] e com o nome do eletrodo.
    """
    # Desenhar o contorno da cabeça (círculo simples)
    head_circle = plt.Circle((0, 0), 1.15, color='lightpink', zorder=0, ec='k')
    ax.add_patch(head_circle)
    
    # Desenhar o nariz: triângulo simétrico posicionado na base do triângulo
    nose = plt.Polygon([[-0.1, 1.15], [0, 1.3], [0.1, 1.15]],
                       closed=True, color='lightpink', edgecolor='k', zorder=3)
    ax.add_patch(nose)
    
    # Desenhar as orelhas
    left_ear = plt.Circle((-1.15, 0), 0.1, color='lightpink', edgecolor='k', zorder=1)
    right_ear = plt.Circle((1.15, 0), 0.1, color='lightpink', edgecolor='k', zorder=1)
    ax.add_patch(left_ear)
    ax.add_patch(right_ear)
    
    # Criar dicionário {Eletrodo: Acurácia}
    cond_values = dict(zip(df['Eletrodo'], df[cond]))
    
    # Preparar listas para o scatter plot
    x_vals = []
    y_vals = []
    colors = []
    
    for elec, (ex, ey) in electrode_positions.items():
        # Obter o valor da acurácia para o eletrodo ou atribui NaN se não houver dado
        valor = cond_values.get(elec, np.nan)
        x_vals.append(ex)
        y_vals.append(ey)
        colors.append(valor)
    
    # Plotar os círculos dos eletrodos
    sc = ax.scatter(x_vals, y_vals, c=colors, cmap='viridis',
                    vmin=0.1905, vmax=0.7619, s=200, edgecolors='k', zorder=2)
    
    # Adicionar o nome de cada eletrodo dentro do círculo
    for elec, (ex, ey) in electrode_positions.items():
        ax.text(ex, ey, elec, ha='center', va='center', fontsize=8, color='black', zorder=4)
    
    # Título estilizado
    ax.set_title(r'$\mathcal{{{}}}$'.format(title_str), fontsize=16, fontweight='bold')
    
    ax.set_aspect('equal', 'box')
    ax.axis('off')
    
    return sc

def main():
    # Ler a planilha Excel e limpar os nomes das colunas
    df = pd.read_excel("Tabela para gráfico dos eletrodos na cabeça.xlsx")
    df.columns = df.columns.str.strip()
    
    # Definição das posições dos eletrodos
    electrode_positions = {
        'Fp1': (-0.3,  1.0),
        'Fp2': ( 0.3,  1.0),
        'F7':  (-0.9,  0.6),
        'F3':  (-0.4,  0.6),
        'Fz':  ( 0.0,  0.6),
        'F4':  ( 0.4,  0.6),
        'F8':  ( 0.9,  0.6),
        'T3':  (-1.0,  0.0),
        'C3':  (-0.4,  0.0),
        'Cz':  ( 0.0,  0.0),
        'C4':  ( 0.4,  0.0),
        'T4':  ( 1.0,  0.0),
        'T5':  (-0.9, -0.6),
        'P3':  (-0.4, -0.6),
        'Pz':  ( 0.0, -0.6),
        'P4':  ( 0.4, -0.6),
        'T6':  ( 0.9, -0.6),
        'O1':  (-0.3, -1.0),
        'O2':  ( 0.3, -1.0),
    }
    
    # Criar figura e eixos para 3 condições (cabeças)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Lista das condições
    conditions = ["(C)", "(F)", "(Q)"]
    scatter_objs = []
    for i, cond in enumerate(conditions):
        ax = axes[i]
        sc = plot_scalp(ax, df, cond, electrode_positions, cond)
        scatter_objs.append(sc)
    
    # Criar um eixo extra para a barra de cores à direita da figura
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(scatter_objs[-1], cax=cbar_ax)
    cbar.set_label("Acurácia", fontsize=12)
    
    # Ajustar layout para evitar sobreposição
    plt.subplots_adjust(left=0.05, right=0.9, wspace=0.3)
    
    # Salvar a figura em um arquivo PNG com alta resolução
    plt.savefig("grafico_eletrodos.png", dpi=300, bbox_inches='tight')
    
    plt.show()

if __name__ == "__main__":
    main()